# CPSA 04 — variant study (activation × training/deployment size)

Trains the variant matrix of YOLOv8n-pose on the mixed dataset: the *curated*
half-bust train list from notebook 01 plus the full-frame images from notebook
02 (`FULL_DATA`). Every row is `(activation, train_imgsz, deploy HxW)`; the
default matrix is:

| variant | activation | train imgsz | deploy input |
| --- | --- | ---: | --- |
| `hswish_640` | Hard-Swish | 640 | 480×640 (baseline) |
| `lrelu_640` | LeakyReLU(0.1015625) | 640 | 480×640 |
| `lrelu_512` | LeakyReLU(0.1015625) | 512 | 384×512 |
| `lrelu_384` | LeakyReLU(0.1015625) | 384 | 288×384 |

* **Urgent run:** `VARIANTS = [DEFAULT_VARIANTS[2]]` (the `lrelu` 512 model
  deployed at 384×512) with `EPOCHS = 30`; the main run uses 120 epochs over the
  full matrix.
* The training list is curated by `cpsa_ml.datasets.curation`: keep crops with
  `height >= MIN_HEIGHT` (128 px), repeat crops that show a hand with >= 10
  visible keypoints `HAND_REPEAT` times (2) and, optionally, crops with >= 5
  visible face keypoints `FACE_REPEAT` times (1 = off). `LABELS_ROOT` points at
  the Task P pseudo-label tree (a dataset named `cpsa-halfbust-pl`), and
  `EXTRA_DATA` mixes in extra `halfbust62` datasets (a dataset named
  `cpsa-hagrid62`) at their own repeat factor.
* The recipe is pinned in `variant_study.AUGMENTATIONS`: explicit SGD
  (`lr0=0.01`, `lrf=0.01`, `momentum=0.937`, `warmup_epochs=3`), `cos_lr=True`,
  `degrees=10`, `close_mosaic=15`, `mixup=0`, `scale=0.5`, `fliplr=0.5` with the
  schema `flip_idx`, Pose loss gain at the Ultralytics default (12.0) and
  `multi_scale=0` (recommendation: off, it costs step time and the crops are
  already scale-augmented). Motion blur
  (`MotionBlur(blur_limit=(3, 9), p=0.15)`) is passed through the `augmentations=`
  train argument next to the Ultralytics albumentations defaults.
* Every row starts from the official COCO `yolov8n-pose.pt`: shape-matching
  backbone/neck tensors and the Pose head's box (`cv2`), class (`cv3`) and `dfl`
  branches are copied; only the keypoint branch `cv4` is freshly initialised
  (`<variant>/transfer_report.json` logs what moved).
* Each row trains square at its own `train_imgsz`, then validates and exports
  `deploy.onnx` at its deployment size. `best.pt` is selected on the full
  half-body val split; `metrics_fp32.json` also reports `metrics.h160` (the
  half-body val crops of at least 160 px, closest to the seated-user demo) and
  one `extra_metrics` section per extra dataset.
* Outputs go to `/kaggle/working/variant_study/<activation>_<train_imgsz>/`:
  `best.pt`, `last.pt`, `results.csv`, `metrics_fp32.json`,
  `transfer_report.json`, `deploy.onnx`, plus the curated lists and
  `curated_train_summary.json` next to them.

The INT8/FP32 scoring runs on the host with `tools/vitis/dump_quant_outputs.py`
and `tools/vitis/eval_quant.py`. The accept/reject rule is the proposal in
`ml/README.md` ("Variant study").

In [ ]:
import subprocess
import sys

# Pin the exact CPSA_2026 commit for a reproducible Kaggle run. The coordinator
# pins the merge commit that contains this notebook and its cpsa_ml changes.
COMMIT_SHA = "a06184367d62061955867d6199b949383e659763"

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        f"git+https://github.com/fcabecciaw/CPSA_2026@{COMMIT_SHA}#subdirectory=ml",
    ],
    check=True,
)

# Motion blur is the only optional augmentation of the study recipe: it needs
# albumentations (Ultralytics 8.4.203 takes custom transforms through the
# `augmentations=` train argument). A failed install is not fatal: the study then
# logs "motion blur is SKIPPED" and records `custom_augmentations: null`.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "albumentations==1.4.21"],
    check=False,
)

In [ ]:
from pathlib import Path

import torch
import ultralytics

from cpsa_ml.datasets.staging import find_inputs, stage_dataset
from cpsa_ml.variant_study import DEFAULT_VARIANTS, StudyConfig, train_study

# Kaggle mounts datasets under /kaggle/input/datasets/<owner>/<slug>/ (older
# kernels: /kaggle/input/<slug>/), possibly with an extra folder level, so the
# dataset roots are located by their data.yaml.
INPUT = Path("/kaggle/input")
DATA = next(
    (
        p
        for p in find_inputs(INPUT, "data.yaml")
        if "cpsa-halfbust" in str(p) and "cpsa-halfbust-pl" not in str(p)
    ),
    None,
)  # the pseudo-label dataset is attached separately as LABELS_ROOT
FULL_DATA = next(
    (p.parent for p in find_inputs(INPUT, "data.yaml") if "cpsa-fullframe" in str(p)), None
)  # notebook-02 dataset; None trains on the half-body set only

# Task P pseudo-labels: a dataset named cpsa-halfbust-pl with a labels/train
# tree; None trains on the ground-truth labels.
LABELS_ROOT = next(
    (
        p.parent.parent
        for p in find_inputs(INPUT, "train")
        if p.parent.name == "labels" and "cpsa-halfbust-pl" in str(p)
    ),
    None,
)

# Extra halfbust62 datasets mixed into the train list as (root, repeat) entries;
# empty when none is attached, so the safety run works without them.
EXTRA_DATA = [(p.parent, 1) for p in find_inputs(INPUT, "data.yaml") if "cpsa-hagrid62" in str(p)]

SCHEMA = "halfbust62"

# Urgent run: the lrelu 512-px model deployed at 384×512 (DEFAULT_VARIANTS[2]).
# Comment this line and uncomment the next one to run the whole matrix instead.
VARIANTS = [DEFAULT_VARIANTS[2]]
# VARIANTS = DEFAULT_VARIANTS

SEED = 0
EPOCHS = 30  # safety run tonight; the main run uses 120
BATCH = 32
TRAIN_DEVICE = "0,1"  # 2x T4 DDP
VAL_DEVICE = "0"
WORKERS = 4
PATIENCE = 30
AMP = True

# Training-list curation (cpsa_ml.datasets.curation).
MIN_HEIGHT = 128
HAND_REPEAT = 2
FACE_REPEAT = 1  # 1 = off

PROJECT = Path("/kaggle/working/runs")
PUBLISH = Path("/kaggle/working/variant_study")

# Official COCO weights, attached to the checkpoints dataset; otherwise
# Ultralytics downloads them (Internet is enabled for this kernel).
COCO_WEIGHTS = next(iter(find_inputs(INPUT, "yolov8n-pose.pt")), Path("yolov8n-pose.pt"))

# Per-row resume, keyed by <activation>_<train_imgsz>:
# {"lrelu_512": "/kaggle/input/cpsa-checkpoints/variant_study/lrelu_512/last.pt"}
RESUME = {}

if DATA is None:
    raise FileNotFoundError("attach fabiocabeccia/cpsa-halfbust (no data.yaml found)")
# Copy the attached datasets to the local disk once: /kaggle/input is a slow
# network mount (about 100 files/s), which would stall the curation scan and
# the image loading of every epoch. The copy is reused if the cell is rerun.
LOCAL_COPY = True
if LOCAL_COPY:
    DATA = stage_dataset(DATA.parent, Path("/tmp/cpsa-halfbust")) / "data.yaml"
    if FULL_DATA is not None:
        FULL_DATA = stage_dataset(FULL_DATA, Path("/tmp/cpsa-fullframe"))
    EXTRA_DATA = [
        (stage_dataset(root, Path("/tmp") / f"extra_{i}"), repeat)
        for i, (root, repeat) in enumerate(EXTRA_DATA)
    ]
print("half-body data:", DATA, "| full-frame data:", FULL_DATA)
print("pseudo-labels:", LABELS_ROOT, "| extra datasets:", EXTRA_DATA)

config = StudyConfig(
    data=DATA,
    full_data=FULL_DATA,
    coco_weights=COCO_WEIGHTS,
    project=PROJECT,
    publish=PUBLISH,
    schema=SCHEMA,
    variants=VARIANTS,
    resume=RESUME,
    seed=SEED,
    epochs=EPOCHS,
    batch=BATCH,
    device=TRAIN_DEVICE,
    val_device=VAL_DEVICE,
    workers=WORKERS,
    patience=PATIENCE,
    amp=AMP,
    min_height=MIN_HEIGHT,
    hand_repeat=HAND_REPEAT,
    face_repeat=FACE_REPEAT,
    labels_root=LABELS_ROOT,
    extra_data=EXTRA_DATA,
)
print("ultralytics", ultralytics.__version__, "torch", torch.__version__)
for entry in config.variants:
    print(
        f"{entry.name}: train {entry.train_imgsz}px, "
        f"deploy {entry.deploy_size[0]}x{entry.deploy_size[1]}"
    )

In [ ]:
summaries = train_study(config)
for summary in summaries:
    print(summary["variant"], "->", summary["run_dir"])
    for kind, path in summary["artifacts"].items():
        print(f"  {kind:22s} {path}")

## Next steps

1. Publish `/kaggle/working/variant_study` as a new version of the
   `fabiocabeccia/cpsa-checkpoints` dataset.
2. On the host, quantize and dump the INT8 head tensors for every variant. Read
   `height` and `width` from the variant's `metrics_fp32.json` (`deploy_size`),
   for example for `hswish_640` (480×640):

   ```bash
   VARIANT=hswish_640
   tools/vitis/run_vitis.sh python tools/vitis/dump_quant_outputs.py \
       --weights variant_study/$VARIANT/best.pt \
       --calib-images train=~/data/cpsa-halfbust/images/train \
       --calib-images train_full=~/data/cpsa-fullframe/images/train \
       --test-images half=~/data/cpsa-halfbust/images/val \
       --test-images full=~/data/cpsa-fullframe/images/val \
       --calib-count 64 --test-count 64 --height 480 --width 640 --threads 1 \
       --out build/variant_study/$VARIANT/quant
   ```

3. Score INT8 and FP32 with the same metric code, against the `hswish_640`
   INT8 baseline:

   ```bash
   PYTHONPATH=src python tools/vitis/eval_quant.py \
       --dump build/variant_study/$VARIANT/quant/quant_outputs.npz \
       --weights variant_study/$VARIANT/best.pt \
       --eval-split half=~/data/cpsa-halfbust/images/val:~/data/cpsa-halfbust/labels/val \
       --eval-split full=~/data/cpsa-fullframe/images/val:~/data/cpsa-fullframe/labels/val \
       --baseline build/variant_study/hswish_640/eval/metrics_int8.json \
       --out build/variant_study/$VARIANT/eval
   ```

   `comparison_splits.md` shows the half and full splits side by side; the
   `framing.*` rows say how many people each split really shows as full body.

4. Apply the proposal rule from `ml/README.md`: a faster activation/input
   pairing is accepted only when its INT8 wrist, hand and face PCK@0.2fw stay
   within two points of the INT8 baseline; `comparison.json` reports the
   per-group decision (`pass` / `fail` / `unknown`).